# S&P 500 Financial Ratio Analysis - Data Preparation

This notebook retrieves financial data (from 2015 to 2024) of 32 representative companies from WRDS Compustat, calculates core financial ratios, and saves them as a CSV file for use by the Streamlit application.

In [1]:
import wrds
import pandas as pd
import numpy as np
from io import StringIO

# Show all columns for easy checking.
pd.set_option('display.max_columns', None)

# Connect to WRDS
db = wrds.Connection()
print("The connection to WRDS was successful.")

Enter your WRDS username [10396]: xunran
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


pgpass file created at C:\Users\10396\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
The connection to WRDS was successful.


## 1. Define the representative company sample

Select 32 leading companies across various industries, covering 11 GICS industry sectors, to ensure the industry representativeness of the sample.

In [8]:
# Construct the ticker_list (32 leading companies across various industries) for the representative sample
ticker_list = [
    'AAPL', 'MSFT', 'NVDA', 'AVGO', 'ADBE',   # Information Technology
    'JPM', 'BAC', 'WFC', 'V',                #  Finance
    'JNJ', 'UNH', 'PFE', 'ABBV',             # Healthcare
    'AMZN', 'TSLA', 'HD',                    # Non-essential Consumer Goods
    'PG', 'KO', 'PEP',                       # Essential Consumer Goods
    'XOM', 'CVX',                            # Energy
    'CAT', 'GE', 'UPS',                      # Industry
    'LIN', 'APD',                            # Materials
    'PLD', 'AMT',                            # Real Estate
    'NEE', 'DUK',                            # Utilities
    'GOOGL', 'META'                          # Communication Services
]
print(f"Representative samples have been set. There are a total of  {len(ticker_list)} companies.")
print("Top 10：", ticker_list[:10])

Representative samples have been set. There are a total of  32 companies.
Top 10： ['AAPL', 'MSFT', 'NVDA', 'AVGO', 'ADBE', 'JPM', 'BAC', 'WFC', 'V', 'JNJ']


## 2. Batch query of Compustat

To avoid the WRDS connection timeout, 10 tickers are queried in each batch, and all batches are then merged together.

In [13]:
# Define the fields to be queried (select only the necessary fields)
columns_query = """
gvkey, datadate, fyr, tic, conm, 
at, lt, seq, ni, revt, act, lct
"""

# The list that stores data for all batches
all_data = []

# Batch-based query (10 companies per batch, adjustable as needed)
batch_size = 10
total_batches = (len(ticker_list) + batch_size - 1) // batch_size

for i in range(0, len(ticker_list), batch_size):
    batch = ticker_list[i:i+batch_size]
    ticker_str = "', '".join(batch)
    query = f"""
    SELECT {columns_query}
    FROM comp.funda
    WHERE tic IN ('{ticker_str}')
        AND datadate >= '2015-01-01'
        AND datadate <= '2024-12-31'
        AND indfmt = 'INDL'
        AND datafmt = 'STD'
        AND popsrc = 'D'
        AND consol = 'C'
    ORDER BY tic, datadate
    """
    print(f"Currently, data for the {i+1} th to {i+len(batch)} th companies is being retrieved（batch {i//batch_size + 1}/{total_batches}）...")
    batch_df = db.raw_sql(query, date_cols=['datadate'])
    all_data.append(batch_df)

# Merge all batches
df_raw = pd.concat(all_data, ignore_index=True)
print(f"Total number of rows in the original data：{len(df_raw)}")

Currently, data for the 1 th to 10 th companies is being retrieved（batch 1/4）...
Currently, data for the 11 th to 20 th companies is being retrieved（batch 2/4）...
Currently, data for the 21 th to 30 th companies is being retrieved（batch 3/4）...
Currently, data for the 31 th to 32 th companies is being retrieved（batch 4/4）...
Total number of rows in the original data：320


## 3. Data cleaning and calculation of financial ratios

- Convert the numerical column to a numeric type
- Handle the missing shareholders' equity (by filling it with the difference between total assets and total liabilities)
- Calculate ROE, profit margin, debt-to-asset ratio, asset turnover ratio, and current ratio.
- Delete invalid rows

In [14]:
# Conversion of numerical columns
numeric_cols = ['at', 'lt', 'seq', 'ni', 'revt', 'act', 'lct']
for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Delete the rows where all the key indicators are empty.
df_clean = df_raw.dropna(subset=['at', 'revt', 'ni'], how='all')

# Shareholder equity fill-in (if seq is missing or 0, use "at - lt" instead)
df_clean['seq_filled'] = df_clean['seq'].fillna(df_clean['at'] - df_clean['lt'])
df_clean['seq_filled'] = df_clean['seq_filled'].where(df_clean['seq_filled'] > 0, 1)

# Calculate the ratio
df_clean['roe'] = df_clean['ni'] / df_clean['seq_filled']
df_clean['profit_margin'] = df_clean['ni'] / df_clean['revt']
df_clean['debt_ratio'] = df_clean['lt'] / df_clean['at']
df_clean['asset_turnover'] = df_clean['revt'] / df_clean['at']
df_clean['current_ratio'] = df_clean['act'] / df_clean['lct']

# Replace infinite values
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

# Extract the year
df_clean['year'] = pd.DatetimeIndex(df_clean['datadate']).year

# Select the final column and rename the "ticker" column
final_df = df_clean[['tic', 'conm', 'year', 'fyr', 
                     'roe', 'profit_margin', 'debt_ratio', 
                     'asset_turnover', 'current_ratio']].copy()
final_df.rename(columns={'tic': 'ticker'}, inplace=True)

# Delete all rows where all the ratios are empty.
final_df.dropna(subset=['roe', 'profit_margin', 'debt_ratio'], how='all', inplace=True)

print(f"The number of valid data rows after cleaning：{len(final_df)}")
print("Data preview：")
final_df.head()

The number of valid data rows after cleaning：320
Data preview：


,ticker,conm,year,fyr,roe,profit_margin,debt_ratio,asset_turnover,current_ratio
0,AAPL,APPLE INC,2015,9,0.447355,0.228458,0.58911,0.804585,1.108771
1,AAPL,APPLE INC,2016,9,0.356237,0.212408,0.601322,0.668636,1.352669
2,AAPL,APPLE INC,2017,9,0.360702,0.210924,0.642845,0.610771,1.276063
3,AAPL,APPLE INC,2018,9,0.555601,0.224341,0.707029,0.72557,1.123843
4,AAPL,APPLE INC,2019,9,0.610645,0.212381,0.732692,0.768572,1.540126


## 4. Save as CSV file

Save the processed data as `sp500_32companies_ratios.csv` for use by the Streamlit application.

In [15]:
final_df.to_csv('sp500_32companies_ratios.csv', index=False)
print("The file has been saved：sp500_32companies_ratios.csv")

The file has been saved：sp500_32companies_ratios.csv


## 5. Streamlit interactive application code (`app.py`)

The following is the complete Streamlit application code, used to present the financial ratio analysis tool. Users can select the company, indicator and year range through the sidebar to dynamically view the line chart and data table.

In [17]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="S&P 500 Financial Analysis Tools", layout="wide")
st.title("📊 Analysis of Financial Ratios of Cross-Industry S&P 500 Companies")
st.markdown("**32 representative companies | 2015-2024** | Quickly compare the profitability, debt-paying ability and operational efficiency")

@st.cache_data
def load_data():
    df = pd.read_csv('sp500_32companies_ratios.csv')
    df['year'] = df['year'].astype(int)
    return df

df = load_data()

st.sidebar.header("⚙️ Filter Settings")
companies = sorted(df['ticker'].unique())
selected_tickers = st.sidebar.multiselect(
    "Select companies (multiple choices are allowed, and it is recommended to choose no more than 5)",
    options=companies,
    default=[companies[0]] if companies else []
)
if len(selected_tickers) > 5:
    st.sidebar.warning("More than 5 options have been selected. Only the top 5 options are displayed.")
    selected_tickers = selected_tickers[:5]

metrics = {
    "ROE": "roe",
    "Profit Margin": "profit_margin",
    "Debt Ratio": "debt_ratio",
    "Asset Turnover": "asset_turnover",
    "Current Ratio": "current_ratio"
}
selected_metrics = st.sidebar.multiselect(
    "Select financial indicators",
    options=list(metrics.keys()),
    default=["ROE", "Debt Ratio"]
)

years = sorted(df['year'].unique())
year_range = st.sidebar.slider(
    "Year range",
    min_value=min(years), max_value=max(years),
    value=(min(years), max(years)), step=1
)

filtered = df[df['ticker'].isin(selected_tickers)]
filtered = filtered[(filtered['year'] >= year_range[0]) & (filtered['year'] <= year_range[1])]

if filtered.empty:
    st.warning("No data available. Please adjust your selection.")
else:
    for metric_display in selected_metrics:
        metric_col = metrics[metric_display]
        plot_df = filtered[['ticker', 'year', metric_col]].dropna()
        if plot_df.empty:
            st.info(f"Indicator {metric_display} has no valid data")
            continue
        fig = px.line(plot_df, x='year', y=metric_col, color='ticker',
                      title=f"{metric_display} Trend Comparison",
                      labels={metric_col: metric_display, 'year': 'year', 'ticker': 'company'},
                      markers=True)
        fig.update_layout(height=450)
        st.plotly_chart(fig, use_container_width=True)
        st.caption(f"💡 {metric_display} The higher the better (the debt-to-asset ratio should be considered in conjunction with the industry) ")

    with st.expander("📋 View the detailed data table"):
        st.dataframe(filtered)

st.markdown("---")
st.caption("Data sources: WRDS Compustat (2015-2024) ")

Writing app.py


## 6. Close the WRDS connection

In [18]:
db.close()

## 7. Obtain the storage location of this file

In [19]:
import os
print(os.getcwd())

C:\Users\10396\Downloads\Python\ACC102 mini assignment
